# Fabric IQ: the RAG literature as a queryable research graph

This notebook queries a Fabric Graph Model built over 20 retrieval-augmented generation
papers. The graph fuses two very different kinds of structure:

| Layer | Source | What it knows |
| --- | --- | --- |
| Citation neighbourhood | Semantic Scholar Graph API | who cites whom, **why** they cite (background / methodology / result), authors, venues |
| Extracted semantics | an LLM reading the indexed full text | what each paper **proposes**, what it **uses**, which **tasks**, **datasets** and **metrics** it works with, and the **limitations** its authors admit to |

A bibliography can tell you that Self-RAG cites DPR. This graph can tell you that it cites
it *methodologically*, that it reuses DPR as a component, that both are evaluated on
Natural Questions, and that both report the same class of limitation.

**Build pipeline**

```
infra/fetch-research-citations.py   -> data/research-graph/citation-graph.json
infra/extract-research-entities.py  -> data/research-graph/paper-entities.json
infra/create-research-graph.py      -> Fabric lakehouse + Graph Model
```


In [1]:
import os

import httpx
from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)

FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_RESEARCH_GRAPH_ID = os.environ["FABRIC_RESEARCH_GRAPH_ID"]

credential = AzureDeveloperCliCredential(tenant_id=FABRIC_TENANT_ID)
fabric_token = credential.get_token("https://api.fabric.microsoft.com/.default").token

GQL_ENDPOINT = (
    "https://api.fabric.microsoft.com/v1/workspaces/"
    f"{FABRIC_WORKSPACE_ID}/GraphModels/{FABRIC_RESEARCH_GRAPH_ID}/executeQuery?preview=true"
)


def gql(query: str, title: str = "", limit: int = 10):
    """Execute a GQL query against the Fabric Graph Model and print the table."""
    response = httpx.post(
        GQL_ENDPOINT,
        headers={"Authorization": f"Bearer {fabric_token}"},
        json={"query": query},
        timeout=180,
    )
    response.raise_for_status()
    body = response.json()

    status = body.get("status", {})
    if status.get("code") != "00000":
        cause = (status.get("cause") or {}).get("description", status.get("description"))
        print(f"GQL ERROR: {cause}")
        return []

    result = body["result"]
    columns = [c["name"] for c in result["columns"]]
    rows = result["data"]

    if title:
        print(f"{title}\n" + "-" * len(title))
    preview = rows[:limit]
    widths = {c: max([len(c), *(len(str(r.get(c, ""))[:60]) for r in preview)]) for c in columns}
    if preview:
        print("  ".join(c.ljust(widths[c]) for c in columns))
        for row in preview:
            print("  ".join(str(row.get(c, ""))[:60].ljust(widths[c]) for c in columns))
    print(f"({len(rows)} rows)\n")
    return rows


print(f"Graph endpoint ready for {FABRIC_RESEARCH_GRAPH_ID}")

Graph endpoint ready for 220c656e-2333-47e4-84fd-1e01d164afaf


## 1. What is actually in the graph

Three tiers of paper. The **core** 20 are the papers whose full text is indexed in Azure AI
Search, so they are the only ones with an extracted semantic layer. **Foundations** are
papers that at least four core papers cite, and **descendants** are papers that cite at
least six core papers. The outer tiers are deliberately thresholded: they exist to give the
core corpus an ancestry and a future, not to be exhaustive.


In [2]:
for label in ["Paper", "Author", "Venue", "Method", "Task", "Dataset", "Metric", "Limitation"]:
    rows = gql(f"MATCH (n:{label}) RETURN COUNT(n) AS n")
    print(f"  {label:12s} {rows[0]['n']:>6} nodes")

for label in ["CITES", "AUTHORED", "COAUTHORED", "PUBLISHED_IN", "PROPOSES",
              "USES", "ADDRESSES", "EVALUATES_ON", "MEASURES", "REPORTS_LIMITATION"]:
    rows = gql(f"MATCH ()-[e:{label}]->() RETURN COUNT(e) AS n")
    print(f"  {label:20s} {rows[0]['n']:>6} edges")

gql("MATCH (p:Paper) RETURN p.tier AS tier, COUNT(p) AS papers GROUP BY tier", "Papers by tier")

n  
119
(1 rows)

  Paper           119 nodes


n   
1077
(1 rows)

  Author         1077 nodes


n 
27
(1 rows)

  Venue            27 nodes


n  
113
(1 rows)

  Method          113 nodes


n 
54
(1 rows)

  Task             54 nodes


...

n 
98
(1 rows)

  EVALUATES_ON             98 edges


n 
95
(1 rows)

  MEASURES                 95 edges


n 
95
(1 rows)

  REPORTS_LIMITATION       95 edges


Papers by tier
--------------
tier        papers
core        20    
foundation  40    
descendant  59    
(3 rows)



[{'tier': 'core', 'papers': 20},
 {'tier': 'foundation', 'papers': 40},
 {'tier': 'descendant', 'papers': 59}]

## 2. Where does this corpus come from?

The shared ancestry of the corpus, ranked by how many of the 20 core papers cite each work.
This is the reading list you would have to absorb before the corpus makes sense — derived
rather than curated.


In [3]:
gql("""
MATCH (core:Paper)-[:CITES]->(f:Paper)
FILTER core.tier = 'core' AND f.tier = 'foundation'
RETURN f.title AS foundation, f.year AS year, COUNT(core) AS cited_by_core
GROUP BY foundation, year
ORDER BY cited_by_core DESC LIMIT 12
""", "Shared intellectual ancestry of the corpus", limit=12)

Shared intellectual ancestry of the corpus
------------------------------------------
foundation                                                    year  cited_by_core
Language Models are Few-Shot Learners                         2020  11           
Natural Questions: A Benchmark for Question Answering Resear  2019  10           
BERT: Pre-training of Deep Bidirectional Transformers for La  2019  9            
PaLM: Scaling Language Modeling with Pathways                 2022  8            
Training language models to follow instructions with human f  2022  8            
REALM: Retrieval-Augmented Language Model Pre-Training        2020  8            
TriviaQA: A Large Scale Distantly Supervised Challenge Datas  2017  7            
The Probabilistic Relevance Framework: BM25 and Beyond        2009  6            
Exploring the Limits of Transfer Learning with a Unified Tex  2019  6            
Finetuned Language Models Are Zero-Shot Learners              2021  6            
Latent Retri

[{'foundation': 'Language Models are Few-Shot Learners',
  'year': 2020,
  'cited_by_core': 11},
 {'foundation': 'Natural Questions: A Benchmark for Question Answering Research',
  'year': 2019,
  'cited_by_core': 10},
 {'foundation': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding',
  'year': 2019,
  'cited_by_core': 9},
 {'foundation': 'PaLM: Scaling Language Modeling with Pathways',
  'year': 2022,
  'cited_by_core': 8},
 {'foundation': 'Training language models to follow instructions with human feedback',
  'year': 2022,
  'cited_by_core': 8},
 {'foundation': 'REALM: Retrieval-Augmented Language Model Pre-Training',
  'year': 2020,
  'cited_by_core': 8},
 {'foundation': 'TriviaQA: A Large Scale Distantly Supervised Challenge Dataset for Reading Comprehension',
  'year': 2017,
  'cited_by_core': 7},
 {'foundation': 'The Probabilistic Relevance Framework: BM25 and Beyond',
  'year': 2009,
  'cited_by_core': 6},
 {'foundation': 'Exploring the Limits o

## 3. Not every citation means the same thing

This is the property a plain bibliography throws away. Semantic Scholar classifies each
citation's **intent**, so we can separate papers that are merely name-checked as background
from papers that are genuinely built upon.

A citation count says a paper is famous. A methodological citation count says it is *useful*.


In [4]:
gql("""
MATCH (src:Paper)-[c:CITES]->(dst:Paper)
FILTER c.isMethodological = true AND dst.tier = 'core'
RETURN dst.title AS cited_paper, COUNT(src) AS methodological_citers
GROUP BY cited_paper
ORDER BY methodological_citers DESC LIMIT 8
""", "Cited as METHODOLOGY, not just background")

gql("""
MATCH (src:Paper)-[c:CITES]->(dst:Paper)
FILTER dst.title = 'Dense Passage Retrieval for Open-Domain Question Answering'
  AND c.intent <> 'uncategorized'
RETURN c.intent AS intent, COUNT(src) AS citers
GROUP BY intent
ORDER BY citers DESC
""", "How the corpus cites DPR specifically")

Cited as METHODOLOGY, not just background
-----------------------------------------
cited_paper                                                   methodological_citers
Dense Passage Retrieval for Open-Domain Question Answering    5                    
Retrieval-Augmented Generation for Knowledge-Intensive NLP T  3                    
Chain of Thought Prompting Elicits Reasoning in Large Langua  2                    
Query Rewriting for Retrieval-Augmented Large Language Model  1                    
Toolformer: Language Models Can Teach Themselves to Use Tool  1                    
Precise Zero-Shot Dense Retrieval without Relevance Labels    1                    
Self-RAG: Learning to Retrieve, Generate, and Critique throu  1                    
Retrieval-Augmented Generation for Large Language Models: A   1                    
(8 rows)



How the corpus cites DPR specifically
-------------------------------------
intent                         citers
background|methodology         3     
methodology                    1     
background|methodology|result  1     
(3 rows)



[{'intent': 'background|methodology', 'citers': 3},
 {'intent': 'methodology', 'citers': 1},
 {'intent': 'background|methodology|result', 'citers': 1}]

## 4. Method lineage

`PROPOSES` and `USES` come from the LLM reading each paper, not from metadata. Joining them
traces a technique from the paper that introduced it to the papers that adopted it — a
dependency graph of ideas.


In [5]:
gql("""
MATCH (origin:Paper)-[:PROPOSES]->(m:Method)<-[:USES]-(adopter:Paper)
RETURN m.name AS method, origin.title AS introduced_by, adopter.title AS adopted_by
ORDER BY method LIMIT 12
""", "Technique lineage: proposed here, reused there", limit=12)

gql("""
MATCH (p:Paper)-[:USES]->(m:Method)
RETURN m.name AS method, COUNT(p) AS papers_using
GROUP BY method
ORDER BY papers_using DESC LIMIT 10
""", "The corpus's shared toolbox")

Technique lineage: proposed here, reused there
----------------------------------------------
method                          introduced_by                                                 adopted_by                                                  
BookCorpus                      The Pile: An 800GB Dataset of Diverse Text for Language Mode  The Pile: An 800GB Dataset of Diverse Text for Language Mode
Chain-of-Thought                Chain of Thought Prompting Elicits Reasoning in Large Langua  ReAct: Synergizing Reasoning and Acting in Language Models  
Chain-of-Thought                Chain of Thought Prompting Elicits Reasoning in Large Langua  Judging LLM-as-a-judge with MT-Bench and Chatbot Arena      
DPR                             Dense Passage Retrieval for Open-Domain Question Answering    Leveraging Passage Retrieval with Generative Models for Open
DPR                             Dense Passage Retrieval for Open-Domain Question Answering    Dense Passage Retrieval for Open-Doma

The corpus's shared toolbox
---------------------------
method      papers_using
BM25        8           
GPT-3.5     5           
GPT-4       5           
DPR         4           
GPT-3       3           
ChatGPT     3           
LLaMA       3           
Contriever  3           
SBERT       3           
FAISS       2           
(10 rows)



[{'method': 'BM25', 'papers_using': 8},
 {'method': 'GPT-3.5', 'papers_using': 5},
 {'method': 'GPT-4', 'papers_using': 5},
 {'method': 'DPR', 'papers_using': 4},
 {'method': 'GPT-3', 'papers_using': 3},
 {'method': 'ChatGPT', 'papers_using': 3},
 {'method': 'LLaMA', 'papers_using': 3},
 {'method': 'Contriever', 'papers_using': 3},
 {'method': 'SBERT', 'papers_using': 3},
 {'method': 'FAISS', 'papers_using': 2}]

## 5. Evaluation practice

Which benchmarks does this literature actually rely on, and for which tasks? This is the
question that decides whether a new result is comparable to anything.


In [6]:
gql("""
MATCH (t:Task)<-[:ADDRESSES]-(p:Paper)-[:EVALUATES_ON]->(d:Dataset)
RETURN t.name AS task, d.name AS dataset, COUNT(p) AS papers
GROUP BY task, dataset
ORDER BY papers DESC LIMIT 10
""", "Task / benchmark pairings")

gql("""
MATCH (p:Paper)-[:MEASURES]->(m:Metric)
RETURN m.name AS metric, COUNT(p) AS papers
GROUP BY metric
ORDER BY papers DESC LIMIT 10
""", "What the field reports")

Task / benchmark pairings
-------------------------
task                  dataset            papers
Question answering    TriviaQA           6     
Question answering    Natural Questions  6     
Question answering    SQuAD              4     
Question answering    WebQuestions       4     
Question answering    HotpotQA           3     
Fact verification     TriviaQA           3     
Question answering    PopQA              3     
Fact verification     HotpotQA           2     
Math problem solving  SVAMP              2     
Fact verification     FEVER              2     
(10 rows)



What the field reports
----------------------
metric            papers
Accuracy          13    
Exact Match       7     
F1                6     
Success Rate      4     
NDCG@10           3     
Recall            3     
Answer Relevance  2     
Hit Rate          2     
Recall@5          2     
ROUGE-L           2     
(10 rows)



[{'metric': 'Accuracy', 'papers': 13},
 {'metric': 'Exact Match', 'papers': 7},
 {'metric': 'F1', 'papers': 6},
 {'metric': 'Success Rate', 'papers': 4},
 {'metric': 'NDCG@10', 'papers': 3},
 {'metric': 'Recall', 'papers': 3},
 {'metric': 'Answer Relevance', 'papers': 2},
 {'metric': 'Hit Rate', 'papers': 2},
 {'metric': 'Recall@5', 'papers': 2},
 {'metric': 'ROUGE-L', 'papers': 2}]

## 6. The corpus's open problems

Every paper buries a limitations paragraph that nobody aggregates. The extraction pulled out
each self-reported limitation and grouped them into recurring themes, so the corpus can be
asked what it collectively admits it cannot do.

This is the query that is impossible without the semantic layer: the answer exists only
inside the prose of twenty separate PDFs.


In [7]:
gql("""
MATCH (p:Paper)-[:REPORTS_LIMITATION]->(l:Limitation)
RETURN l.theme AS recurring_limitation, COUNT(p) AS statements
GROUP BY recurring_limitation
ORDER BY statements DESC LIMIT 15
""", "What this literature admits it cannot do", limit=15)

What this literature admits it cannot do
----------------------------------------
recurring_limitation                                      statements
Retrieval quality bounds end-task accuracy                17        
Limited generalization, transfer, and data coverage       11        
LLM-as-judge and benchmark bias                           9         
Efficiency and scalability costs                          9         
Hallucination, grounding, and weak synthesis              8         
Evaluation ambiguity and immature measurement             8         
Architectural rigidity and lack of interactive retrieval  8         
Safety, security, and oversight gaps                      7         
Agent and workflow instability                            6         
Tool-use limitations and sample inefficiency              5         
Reasoning reliability and process fidelity                4         
Prompt and instruction sensitivity                        2         
Other                

[{'recurring_limitation': 'Retrieval quality bounds end-task accuracy',
  'statements': 17},
 {'recurring_limitation': 'Limited generalization, transfer, and data coverage',
  'statements': 11},
 {'recurring_limitation': 'LLM-as-judge and benchmark bias', 'statements': 9},
 {'recurring_limitation': 'Efficiency and scalability costs', 'statements': 9},
 {'recurring_limitation': 'Hallucination, grounding, and weak synthesis',
  'statements': 8},
 {'recurring_limitation': 'Evaluation ambiguity and immature measurement',
  'statements': 8},
 {'recurring_limitation': 'Architectural rigidity and lack of interactive retrieval',
  'statements': 8},
 {'recurring_limitation': 'Safety, security, and oversight gaps',
  'statements': 7},
 {'recurring_limitation': 'Agent and workflow instability', 'statements': 6},
 {'recurring_limitation': 'Tool-use limitations and sample inefficiency',
  'statements': 5},
 {'recurring_limitation': 'Reasoning reliability and process fidelity',
  'statements': 4},
 

In [8]:
gql("""
MATCH (p:Paper)-[:REPORTS_LIMITATION]->(l:Limitation)
FILTER l.theme = 'Retrieval quality bounds end-task accuracy'
RETURN p.title AS paper, l.statement AS in_the_authors_own_words
LIMIT 8
""", "Drilling into the most common admission", limit=8)

Drilling into the most common admission
---------------------------------------
paper                                                         in_the_authors_own_words                                    
Task-aware Retrieval with Instructions                        TART-dual shows performance deterioration on some datasets s
Retrieval-Augmented Generation for Large Language Models: A   A single retrieval based on the original query may be insuff
Retrieval-Augmented Generation for Large Language Models: A   The presence of noise or contradictory information during re
Retrieval-Augmented Generation for Large Language Models: A   Naive RAG retrieval often struggles with precision and recal
Dense Passage Retrieval for Open-Domain Question Answering    DPR may lack sufficient capacity to represent salient rare p
Dense Passage Retrieval for Open-Domain Question Answering    DPR underperforms BM25 on SQuAD, which the authors attribute
RAG-Fusion: a New Take on Retrieval-Augmented Generation   

[{'paper': 'Task-aware Retrieval with Instructions',
  'in_the_authors_own_words': 'TART-dual shows performance deterioration on some datasets such as Touche-2020 and Climate-FEVER.'},
 {'paper': 'Retrieval-Augmented Generation for Large Language Models: A Survey',
  'in_the_authors_own_words': 'A single retrieval based on the original query may be insufficient for complex issues requiring adequate context information.'},
 {'paper': 'Retrieval-Augmented Generation for Large Language Models: A Survey',
  'in_the_authors_own_words': 'The presence of noise or contradictory information during retrieval can harm output quality, so robustness remains a challenge.'},
 {'paper': 'Retrieval-Augmented Generation for Large Language Models: A Survey',
  'in_the_authors_own_words': 'Naive RAG retrieval often struggles with precision and recall, causing irrelevant chunk selection and missing crucial information.'},
 {'paper': 'Dense Passage Retrieval for Open-Domain Question Answering',
  'in_the_au

## 7. Where the evidence runs out

Coverage matrix of method against benchmark. A filled cell means at least one paper in the
corpus uses that method *and* evaluates on that benchmark. An empty cell means this corpus
contains no evidence about that combination.

Read it as a scope statement, not a claim about the whole field: it describes exactly what
these twenty papers do and do not establish.


In [9]:
import collections

uses = gql("MATCH (p:Paper)-[:USES]->(m:Method) RETURN m.name AS entity, p.title AS paper", limit=0)
evaluations = gql("MATCH (p:Paper)-[:EVALUATES_ON]->(d:Dataset) RETURN d.name AS entity, p.title AS paper", limit=0)


def papers_by(rows, minimum):
    """Group papers by entity name, keeping only entities the corpus uses repeatedly."""
    grouped = collections.defaultdict(set)
    for row in rows:
        grouped[row["entity"]].add(row["paper"])
    ranked = sorted(grouped.items(), key=lambda item: -len(item[1]))
    return {name: papers for name, papers in ranked if len(papers) >= minimum}


methods = papers_by(uses, 3)
datasets = papers_by(evaluations, 3)

width = max(len(name) for name in methods)
print(f"{'':{width}}  " + "  ".join(f"{i:>2}" for i in range(1, len(datasets) + 1)))
for method, method_papers in methods.items():
    cells = "  ".join(" *" if method_papers & datasets[d] else " ." for d in datasets)
    print(f"{method:{width}}  {cells}")

print()
for index, (dataset, dataset_papers) in enumerate(datasets.items(), start=1):
    print(f"  {index:>2} = {dataset} ({len(dataset_papers)} papers)")

filled = sum(1 for m in methods.values() for d in datasets.values() if m & d)
total = len(methods) * len(datasets)
print(f"\n{filled} of {total} method/benchmark cells carry evidence in this corpus.")
print("Empty cells are not refutations - they are unwritten papers.")

(110 rows)



(98 rows)

             1   2   3   4   5   6
BM25         *   *   *   *   *   *
GPT-3.5      *   .   .   .   .   .
GPT-4        *   *   .   .   .   *
DPR          *   *   *   *   .   .
Contriever   .   *   .   .   .   *
LLaMA        .   *   .   .   .   *
ChatGPT      .   .   .   .   *   *
GPT-3        .   .   .   .   *   .
SBERT        *   .   .   .   .   .

   1 = Natural Questions (6 papers)
   2 = TriviaQA (6 papers)
   3 = WebQuestions (4 papers)
   4 = SQuAD (4 papers)
   5 = HotpotQA (3 papers)
   6 = PopQA (3 papers)

22 of 54 method/benchmark cells carry evidence in this corpus.
Empty cells are not refutations - they are unwritten papers.


## 8. Who is building on this corpus now

Descendants are papers that cite at least six of the twenty core papers, so they are
attempting to synthesize the corpus rather than cite one part of it. Their titles are a
fair proxy for where the field went next.


In [10]:
gql("""
MATCH (d:Paper)-[:CITES]->(core:Paper)
FILTER d.tier = 'descendant' AND core.tier = 'core'
RETURN d.title AS descendant, d.year AS year, COUNT(core) AS core_papers_cited
GROUP BY descendant, year
ORDER BY core_papers_cited DESC LIMIT 10
""", "Recent work synthesizing this corpus", limit=10)

Recent work synthesizing this corpus
------------------------------------
descendant                                                    year  core_papers_cited
PaperClaw: Harnessing Agents for Autonomous Research and Hum  2026  11               
From Question Answering to Task Completion: A Survey on Agen  2026  11               
A corrective agentic hybrid RAG and an operations-grounded e  2026  9                
AB-RAG: Adaptive Budgeted Retrieval-Augmented Generation for  2026  9                
SkillCenter: A Large-Scale Source-Grounded Skill Library for  2026  8                
From Prompts to Contracts: Harness Engineering for Auditable  2026  8                
LLM-Guided Planning for Multi-hop Reasoning over Multimodal   2026  8                
From BM25 to Corrective RAG: Benchmarking Retrieval Strategi  2026  8                
Beyond the Reranker: Do RAG Retrieval Enhancements Help Once  2026  8                
Decoupling Search from Reasoning: A Vendor-Agnostic Groundin  2026

[{'descendant': 'PaperClaw: Harnessing Agents for Autonomous Research and Human-in-the-Loop Refinement',
  'year': 2026,
  'core_papers_cited': 11},
 {'descendant': 'From Question Answering to Task Completion: A Survey on Agent System and Harness Design',
  'year': 2026,
  'core_papers_cited': 11},
 {'descendant': 'A corrective agentic hybrid RAG and an operations-grounded evaluation for a scientific facility',
  'year': 2026,
  'core_papers_cited': 9},
 {'descendant': 'AB-RAG: Adaptive Budgeted Retrieval-Augmented Generation for Reliable Question Answering',
  'year': 2026,
  'core_papers_cited': 9},
 {'descendant': 'SkillCenter: A Large-Scale Source-Grounded Skill Library for Autonomous AI Agents',
  'year': 2026,
  'core_papers_cited': 8},
 {'descendant': 'From Prompts to Contracts: Harness Engineering for Auditable Enterprise LLM Agents',
  'year': 2026,
  'core_papers_cited': 8},
 {'descendant': 'LLM-Guided Planning for Multi-hop Reasoning over Multimodal Nuclear Regulatory Docume

## 9. Graph and knowledge base together

The two systems answer different halves of a question and neither can do the other's job.

- The **graph** knows structure: which paper is load-bearing, which technique came from where.
- The **knowledge base** knows content: what a paper actually argues, with citations.

Here the graph chooses the target and the knowledge base explains it, so the question asked
of Foundry IQ was itself derived from the data rather than written by hand.


In [11]:
import time

from azure.identity import DefaultAzureCredential
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
KNOWLEDGE_BASE_NAME = "research-literature-kb"
PAPERS_KS = "arxiv-papers-knowledge-source"

search_credential = DefaultAzureCredential()
kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
)

# Step 1 - the GRAPH picks the target: the core paper other papers reuse
# methodologically most often, plus the limitation theme it reports.
selection = gql("""
MATCH (src:Paper)-[c:CITES]->(dst:Paper)
FILTER c.isMethodological = true AND dst.tier = 'core'
RETURN dst.title AS paper, COUNT(src) AS methodological_citers
GROUP BY paper
ORDER BY methodological_citers DESC LIMIT 1
""", "Step 1 - graph selects the most methodologically reused paper")

target = selection[0]["paper"]

weaknesses = gql(
    "MATCH (p:Paper)-[:REPORTS_LIMITATION]->(l:Limitation) "
    f"FILTER p.title = '{target}' "
    "RETURN l.theme AS theme, l.statement AS statement LIMIT 3",
    "Step 2 - graph retrieves what its authors admit",
)

# Step 3 - the KNOWLEDGE BASE explains the paper and confronts it with its own
# reported weakness. Both halves of the question came from the graph.
admission = weaknesses[0]["statement"] if weaknesses else ""
question = (
    f"According to the paper titled '{target}', what does it propose and what did it "
    f"demonstrate? The authors also report this limitation: \"{admission}\" "
    "Explain how that limitation constrains the contribution. Answer only from that paper."
)
print(f"Step 3 - knowledge base question:\n  {question}\n")

request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user", content=[KnowledgeBaseMessageTextContent(text=question)]
        )
    ],
    knowledge_source_params=[
        FileKnowledgeSourceParams(
            knowledge_source_name=PAPERS_KS,
            include_references=True,
            include_reference_source_data=True,
        )
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

started = time.time()
result = kb_client.retrieve(retrieval_request=request)
print(f"(retrieved in {time.time() - started:.1f}s)\n")

display(Markdown(result.response[0].content[0].text))

Step 1 - graph selects the most methodologically reused paper
-------------------------------------------------------------
paper                                                       methodological_citers
Dense Passage Retrieval for Open-Domain Question Answering  5                    
(1 rows)



Step 2 - graph retrieves what its authors admit
-----------------------------------------------
theme                                                statement                                                   
Limited generalization, transfer, and data coverage  Direct cross-dataset application causes a 3–5 point drop in 
Retrieval quality bounds end-task accuracy           DPR may lack sufficient capacity to represent salient rare p
Efficiency and scalability costs                     Building the dense index is much more expensive than sparse 
(3 rows)

Step 3 - knowledge base question:
  According to the paper titled 'Dense Passage Retrieval for Open-Domain Question Answering', what does it propose and what did it demonstrate? The authors also report this limitation: "Direct cross-dataset application causes a 3–5 point drop in top-20 retrieval accuracy relative to the best fine-tuned model." Explain how that limitation constrains the contribution. Answer only from that paper.



(retrieved in 12.6s)



The paper proposes a **dense retrieval method for open-domain QA** that uses **dense representations alone** for passage retrieval, with **question and passage embeddings learned from a small number of question–passage pairs using a simple dual-encoder framework**.[ref_id:1] More specifically, the authors say they ask whether one can train a better dense embedding model **using only pairs of questions and passages (or answers), without additional pretraining**, and their final method optimizes **inner products of question and passage vectors** with an objective that compares **all question–passage pairs in a batch**.[ref_id:3][ref_id:15]

The paper demonstrates that this approach, called **Dense Passage Retriever (DPR)**, can **greatly outperform BM25/Lucene-style sparse retrieval** on open-domain QA retrieval benchmarks.[ref_id:1][ref_id:3] In the abstract, the authors report that DPR beats a strong Lucene-BM25 system by **9%–19% absolute in top-20 passage retrieval accuracy** across a wide range of open-domain QA datasets.[ref_id:1] They also demonstrate that better retrieval translates into better QA performance: the paper says DPR **helps the end-to-end QA system establish new state of the art on multiple open-domain QA benchmarks**.[ref_id:1] In their contribution statement, they frame this as showing both that **simple fine-tuning of question and passage encoders on existing question-passage pairs is sufficient to greatly outperform BM25** and that **higher retrieval precision indeed translates to higher end-to-end QA accuracy**.[ref_id:3]

The paper also gives a concrete example of what it demonstrated on Natural Questions: DPR achieved **65.2% Top-5 retrieval accuracy versus 42.9% for BM25**, and this led to **41.5% end-to-end QA accuracy versus 33.3% for ORQA** in the open Natural Questions setting.[ref_id:3]

Regarding the limitation, the paper states that when DPR is **trained on Natural Questions only** and then **applied directly to WebQuestions and CuratedTREC without additional fine-tuning**, it suffers a **3–5 point loss relative to the best fine-tuned model in top-20 retrieval accuracy**: **69.9 vs. 75.0 on WebQuestions** and **86.3 vs. 89.1 on TREC**.[ref_id:2] The authors still say this means DPR **generalizes well**, because even without target-dataset fine-tuning it remains well above the BM25 baseline, which is **55.0 on WebQuestions** and **70.9 on TREC**.[ref_id:2]

That limitation constrains the contribution because it shows the paper does **not** establish that DPR reaches the **best retrieval performance out of the box across datasets**.[ref_id:2] Instead, the strongest results still depend on **dataset-specific fine-tuning**, since direct transfer is measurably worse than the best fine-tuned model by **3–5 top-20 accuracy points**.[ref_id:2] So the contribution is strongest as a demonstration that a **simple dense retriever can outperform BM25 and improve QA**, and that it **transfers reasonably well**, but not as a claim that one trained DPR model fully eliminates the need for **additional tuning on new datasets**.[ref_id:2][ref_id:3]

## Notes for anyone re-running this

### Fabric GQL constraints that cost real debugging time

- Use `FILTER`, not Cypher's `WHERE`. The endpoint frequently **times out** on `WHERE`
  rather than returning a syntax error, so the mistake looks like a hang.
- `GROUP BY` takes the `RETURN` alias, not the property path. `GROUP BY dst.title` fails
  with `mismatched input '.'`; write `RETURN dst.title AS paper ... GROUP BY paper`.
  There is no implicit grouping.
- `EXISTS` with a graph pattern is **not supported yet**, so anti-joins ("never combined
  with") have to be computed client-side from two queries. That is what section 7 does.
- `text` is a reserved word and cannot be used as a property name. The `Limitation` node
  uses `statement` instead.
- Property types are `STRING`, `INT`, `FLOAT`, `BOOLEAN`, `DATETIME`. `BOOL` is rejected.

### How much of this is machine-generated, and how far to trust it

- Citation edges, intents and the `isInfluential` flag come from Semantic Scholar. Intent
  labels are only present on about half the outbound references and almost never on
  inbound citations, which is why section 3 filters `intent <> 'uncategorized'` instead of
  reporting a percentage.
- Methods, tasks, datasets, metrics and limitations were extracted by an LLM from the
  indexed full text, then canonicalized in a second pass. The extraction is good but not
  perfect: a few benchmarks (ALFWorld, APIBench) land under `Method` because the papers
  discuss them as harnesses. Treat the semantic layer as a strong index, not as ground truth.
- The corpus is 20 papers. Every count in this notebook describes *this corpus*, not the
  field. Section 7 is deliberately worded that way.
